# Titanic Dataset Analysis

## Assignment 2: Titanic Survival Prediction
This notebook contains the complete workflow for data cleaning, feature engineering, and feature selection on the Titanic dataset.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/train.csv')
df.head()

## Part 1: Data Cleaning
### 1. Missing Value Handling

In [ ]:
# Missing values heatmap (before cleaning)
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap (Before Cleaning)')
plt.show()

print("Missing Values:\n", df.isnull().sum())

In [ ]:
# Run data cleaning script
import sys
sys.path.append('../scripts')
import data_cleaning  # type: ignore

df_cleaned = data_cleaning.clean_data('../data/train.csv', None)
print("Missing Values After Cleaning:\n", df_cleaned.isnull().sum())

### 2. Outlier Handling Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(y=df['Age'], ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Age Before Outlier Handling')

sns.boxplot(y=df_cleaned['Age'], ax=axes[0,1], color='lightgreen')
axes[0,1].set_title('Age After IQR Capping')

sns.boxplot(y=df['Fare'], ax=axes[1,0], color='lightcoral')
axes[1,0].set_title('Fare Before Outlier Handling')

sns.boxplot(y=df_cleaned['Fare'], ax=axes[1,1], color='gold')
axes[1,1].set_title('Fare After 99th Percentile Capping')

plt.tight_layout()
plt.show()

## Part 2: Feature Engineering
Running the engineering script to derive Title, Deck, AgeGroup, apply One-Hot Encoding, scaling, and Log transformations.

In [ ]:
import feature_engineering  # type: ignore

# Note: feature_engineering.py expects clean data, but drops 'Name'.
# We'll use the script to process it, but we can visualize with a pre-encoded version for EDA.

df_engineered = feature_engineering.engineer_features(df_cleaned)
df_engineered.head()

In [ ]:
# Visualizations for Feature Engineering
# To plot Title and AgeGroup, we'll re-derive them temporarily since the script one-hot encoded them.
temp_df = feature_engineering.create_derived_features(df_cleaned)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x='Title', y='Survived', data=temp_df, ax=axes[0], ci=None)
axes[0].set_title('Survival Rate by Title')

sns.barplot(x='AgeGroup', y='Survived', data=temp_df, ax=axes[1], ci=None)
axes[1].set_title('Survival Rate by AgeGroup')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_cleaned['Fare'], bins=30, kde=True, ax=axes[0], color='purple')
axes[0].set_title('Distribution of Fare Before Log Transform')

# Extracting the log transformed column which is standard scaled, 
# so we will manually log it for the plot to show the pure log distribution.
fare_log = np.log1p(df_cleaned['Fare'])
sns.histplot(fare_log, bins=30, kde=True, ax=axes[1], color='orange')
axes[1].set_title('Distribution of Fare After Log Transform')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_cleaned['Age'], bins=30, kde=True, ax=axes[0], color='blue')
axes[0].set_title('Distribution of Age Before Transformation')

# Feature Engineering script scales Age. We show the distribution after standard scaling.
sns.histplot(df_engineered['Age'], bins=30, kde=True, ax=axes[1], color='cyan')
axes[1].set_title('Distribution of Age After Standardization')

plt.tight_layout()
plt.show()

## Part 3: Feature Selection
### 1. Correlation Analysis

In [ ]:
plt.figure(figsize=(16, 12))
corr_matrix = df_engineered.corr()
sns.heatmap(corr_matrix, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap (After Encoding)')
plt.show()

### 2. Feature Importance & RFECV

In [ ]:
import feature_selection  # type: ignore

# We'll do this manually here to extract the model for the plot
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE

df_reduced, to_drop = feature_selection.remove_highly_correlated_features(df_engineered)

X = df_reduced.drop(['Survived', 'PassengerId'], axis=1, errors='ignore')
y = df_reduced['Survived']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x=importances.values, y=importances.index)
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

In [ ]:
n_features = min(12, X.shape[1])
rfe = RFE(estimator=rf, n_features_to_select=n_features, step=1)
rfe.fit(X, y)

selected_features = X.columns[rfe.support_].tolist()
dropped_features = X.columns[~rfe.support_].tolist()
print("--- Justification for Feature Selection ---\n")
print("Features Dropped during Correlation Analysis (>0.8 threshold):")
for feat in to_drop:
    print(f" - {feat}: Dropped due to high redundancy/correlation with other features.")

print("\nFeatures Dropped by RFE (Low Importance):")
for feat in dropped_features:
    print(f" - {feat}: Dropped because RFE deemed it less predictive compared to the top {n_features} features.")

print("\nFeatures Kept by RFE (Top Predictors):")
for feat in selected_features:
    print(f" - {feat}: Kept because Random Forest importance and RFE selected it as a strong predictor.")

### 3. Pairplot of Top Features

In [ ]:
# Pairplot of top 5 features colored by Survived
top_5_features = importances.index[:5].tolist()
plot_df = df_reduced[top_5_features + ['Survived']]

sns.pairplot(plot_df, hue='Survived', palette='seismic')
plt.suptitle('Pairplot of Top 5 Features', y=1.02)
plt.show()